In [3]:
import pandas as pd
import joblib
import random
import shap
from sklearn.metrics import f1_score

# Load saved elements
test_df = pd.read_csv('test_split.csv').dropna()
tfidf = joblib.load('tfidf_vectorizer.pkl')
lr_model = joblib.load('logistic_regression_model.pkl')

def inject_visual_homoglyphs(text, perturbation_probability=0.20):
    homoglyphs = {
        'a': ['4', '@', 'α'], 'e': ['3', '€'], 'i': ['1', '!', '|'],
        'o': ['0', 'ø'], 's': ['5', '$'], 't': ['7', '+']
    }
    words = str(text).split()
    perturbed_words = []
    for word in words:
        perturbed_word = ""
        for char in word:
            if char.lower() in homoglyphs and random.random() < perturbation_probability:
                perturbed_word += random.choice(homoglyphs[char.lower()])
            else:
                perturbed_word += char
        perturbed_words.append(perturbed_word)
    return " ".join(perturbed_words)

# Ensure matching keys
X_test_clean = test_df['text']
y_test = test_df['label']
X_test_adversarial = X_test_clean.apply(inject_visual_homoglyphs)

X_test_clean_tfidf = tfidf.transform(X_test_clean)
X_test_adv_tfidf = tfidf.transform(X_test_adversarial)

c:\Users\BLACKBOX\.anaconda-desktop\micromamba\envs\cuda\envs\condaenv1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Clean Eval
preds_clean = lr_model.predict(X_test_clean_tfidf)
f1_clean = f1_score(y_test, preds_clean)

# Adversarial Eval
preds_adv = lr_model.predict(X_test_adv_tfidf)
f1_adv = f1_score(y_test, preds_adv)

print("--- Robustness Performance Gap ---")
print(f"Clean Dataset F1 Score:        {f1_clean:.4f}")
print(f"Adversarial Homoglyph F1 Score: {f1_adv:.4f}")
print(f"Performance Drop Percentage:    {((f1_clean - f1_adv) / f1_clean) * 100:.2f}%")

--- Robustness Performance Gap ---
Clean Dataset F1 Score:        0.5516
Adversarial Homoglyph F1 Score: 0.4323
Performance Drop Percentage:    21.62%


In [5]:
print("Initializing SHAP Explainer...")
# Explain predictions on a small slice to observe token shifting
explain_sample = X_test_clean.iloc[:5]
explain_sample_adv = X_test_adversarial.iloc[:5]

# Fit background distribution
explainer = shap.LinearExplainer(lr_model, tfidf.transform(test_df['text'].iloc[:200]))

# Generate feature importance distributions
shap_values_clean = explainer.shap_values(tfidf.transform(explain_sample))
shap_values_adv = explainer.shap_values(tfidf.transform(explain_sample_adv))

print("SHAP values calculated.")
print("Compare 'shap_values_clean' arrays against 'shap_values_adv' to show feature shifting in your manuscript figures.")

Background dataset has 200 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=200 when initializing the masker.


Initializing SHAP Explainer...
SHAP values calculated.
Compare 'shap_values_clean' arrays against 'shap_values_adv' to show feature shifting in your manuscript figures.


In [ ]:
# ==========================================
# CELL 4: LINEAR SVC (SVM) ADVERSARIAL EVALUATION
# ==========================================
import joblib
from sklearn.metrics import f1_score

print("Loading saved Linear SVC (SVM) model...")
# Load the SVM model saved from your baseline notebook
svc_model = joblib.load('linear_svc_model.pkl')

# 1. Predict on Clean Text Matrix
preds_clean_svc = svc_model.predict(X_test_clean_tfidf)
f1_clean_svc = f1_score(y_test, preds_clean_svc)

# 2. Predict on Adversarial (Homoglyph) Text Matrix
preds_adv_svc = svc_model.predict(X_test_adv_tfidf)
f1_adv_svc = f1_score(y_test, preds_adv_svc)

# 3. Calculate Performance Degradation
drop_svc = ((f1_clean_svc - f1_adv_svc) / f1_clean_svc) * 100

print("\n=== LINEAR SVC (SVM) OBSERVATIONS ===")
print(f"Clean Text F1-Score:        {f1_clean_svc:.4f}")
print(f"Adversarial Text F1-Score:  {f1_adv_svc:.4f}")
print(f"Performance Drop:           {drop_svc:.2f}%")

In [ ]:
# ==========================================
# CELL 7: SHAP EXPLAINABILITY FOR LINEAR SVC (SVM)
# ==========================================
import shap
import numpy as np

print("Initializing SHAP LinearExplainer for SVM...")

# 1. Initialize the SHAP explainer specifically for linear structures
# We use the clean TF-IDF matrix background sample to remain consistent with your LR setup
svc_explainer = shap.LinearExplainer(svc_model, X_test_clean_tfidf_sample)

# 2. Compute SHAP force values for your sample text slices
# (Assumes 'X_test_clean_sample' and 'X_test_adv_sample' match your LR SHAP cell setup)
shap_values_svc_clean = svc_explainer(X_test_clean_tfidf_sample)
shap_values_svc_adv = svc_explainer(X_test_adv_tfidf_sample)

print("SVM SHAP values calculated successfully!")

# 3. Extract feature names to verify token attribution drop
feature_names = tfidf.get_feature_names_out()

# Locate the index of a specific toxic word you tested in your LR SHAP cell 
# Replace 'badword' with the exact toxic token string used in your previous cells
target_word = "badword" 

if target_word in feature_names:
    word_idx = np.where(feature_names == target_word)[0][0]
    
    # Extract absolute mathematical force assigned to this feature
    clean_force = np.abs(shap_values_svc_clean.values[:, word_idx]).mean()
    adv_force = np.abs(shap_values_svc_adv.values[:, word_idx]).mean()
    
    print(f"\n=== SVM SHAP TRACKING ANALYSIS FOR '{target_word}' ===")
    print(f"Clean Token Influence Force:       {clean_force:.6f}")
    print(f"Adversarial Token Influence Force: {adv_force:.6f} -> (Should hit 0.000000)")
else:
    print(f"\nToken '{target_word}' not found in TF-IDF vocabulary.")

# 4. Generate local text plots directly inside your notebook cells
print("\nDisplaying SVM local feature attribution graphs...")
shap.plots.text(shap_values_svc_clean[0])
shap.plots.text(shap_values_svc_adv[0])

In [ ]:
# ==========================================
# CELL 5: DISTILBERT TRANSFORMER ADVERSARIAL EVALUATION
# ==========================================
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset

print("Loading fine-tuned DistilBERT model and context-aware tokenizer to GPU...")
# Note: Ensure './results' points to your saved checkpoint folder from Notebook 2
model_path = "./results" 
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained(model_path).to("cuda")

# 1. Package your clean and adversarial strings into a Hugging Face Dataset format
eval_df = pd.DataFrame({"text": X_test_adversarial, "label": y_test})
eval_dataset = Dataset.from_pandas(eval_df)

# 2. Tokenize using the deep sub-word network rules
def tokenize_maps(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=64)

tokenized_eval = eval_dataset.map(tokenize_maps, batched=True)
tokenized_eval.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# 3. Process inference via an iterative PyTorch DataLoader
loader = torch.utils.data.DataLoader(tokenized_eval, batch_size=32)
model.eval()

predictions, true_labels = [], []
with torch.no_grad():
    for batch in loader:
        inputs = batch["input_ids"].to("cuda")
        masks = batch["attention_mask"].to("cuda")
        outputs = model(inputs, attention_mask=masks)
        
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        predictions.extend(preds)
        true_labels.extend(batch["label"].numpy())

# 4. Compute Final Transformer Robustness Accuracy
f1_adv_distilbert = f1_score(true_labels, predictions)
drop_distilbert = ((0.8361 - f1_adv_distilbert) / 0.8361) * 100

print("\n=== DISTILBERT TRANSFORMER OBSERVATIONS ===")
print(f"Clean Text F1-Score (Notebook 2 baseline): 0.8361")
print(f"Adversarial Text F1-Score:                 {f1_adv_distilbert:.4f}")
print(f"Performance Drop:                          {drop_distilbert:.2f}%")

In [ ]:
# ==========================================
# CELL 6: MANUSCRIPT MASTER DATA COMPILATION
# ==========================================
# Safely capture your original cell 2 variables
try:
    lr_clean = f1_score_clean
    lr_adv = f1_score_adv
    lr_drop = drop_percentage
except NameError:
    # Hardcoded fallback values from your original Cell 2 run outputs
    lr_clean, lr_adv, lr_drop = 0.5516, 0.4323, 21.62

# Construct the central dataframe
master_summary = pd.DataFrame({
    "Model Architecture": ["Logistic Regression", "Linear SVC (SVM)", "DistilBERT (Transformer)"],
    "Feature Extraction": ["TF-IDF (Word Count)", "TF-IDF (Word Count)", "WordPiece (Contextual)"],
    "Clean F1-Score": [f"{lr_clean:.4f}", f"{f1_clean_svc:.4f}", "0.8361"],
    "Adversarial F1-Score": [f"{lr_adv:.4f}", f"{f1_adv_svc:.4f}", f"{f1_adv_distilbert:.4f}"],
    "Performance Impact (%)": [f"-{lr_drop:.2f}%", f"-{drop_svc:.2f}%", f"-{drop_distilbert:.2f}%"]
})

print("\n" + "="*65)
print("       PUBLICATION-READY MASTER METRIC ATTRIBUTION MATRIX")
print("="*65)
print(master_summary.to_string(index=False))
print("="*65)

In [ ]:
# ==========================================================
# CELL 11: GRANULAR ADVERSARIAL ERROR LIFECYCLE BREAKDOWN
# ==========================================================
from sklearn.metrics import classification_report, confusion_matrix

print("Processing granular confusion matrices across all model architectures...")

# 1. Extract Classification Reports for Toxic Class (Label = 1)
report_lr = classification_report(y_test, preds_adv, output_dict=True)
report_svc = classification_report(y_test, preds_adv_svc, output_dict=True)

# DistilBERT predictions evaluation 
# (Using the predictions array calculated inside your active Cell 5/6 memory)
report_db = classification_report(true_labels, predictions, output_dict=True)

# 2. Extract Confusion Matrices to calculate exact mistake metrics
cm_lr = confusion_matrix(y_test, preds_adv)
cm_svc = confusion_matrix(y_test, preds_adv_svc)
cm_db = confusion_matrix(true_labels, predictions)

# Organize a highly detailed metrics table
detailed_observations = pd.DataFrame({
    "Architecture": ["Logistic Regression", "Linear SVC (SVM)", "DistilBERT (Transformer)"],
    "Toxic Class Precision": [report_lr['1']['precision'], report_svc['1']['precision'], report_db['1']['precision']],
    "Toxic Class Recall": [report_lr['1']['recall'], report_svc['1']['recall'], report_db['1']['recall']],
    "False Negatives (Missed Trolls)": [cm_lr[1][0], cm_svc[1][0], cm_db[1][0]],
    "False Positives (False Alarms)": [cm_lr[0][1], cm_svc[0][1], cm_db[0][1]]
})

print("\n" + "="*80)
print("              CRITICAL ATTACK FAILURE BREAKDOWN MATRIX")
print("="*80)
print(detailed_observations.to_string(index=False))
print("="*80)

In [ ]:
# ==========================================================
# CELL 12: MATHEMATICAL INTERPRETABILITY INSTABILITY METRICS
# ==========================================================
print("Calculating Global SHAP Stability Drift Indices...")

# Measure the mean absolute variance between clean and attacked token distributions
# This yields the exact rate at which feature reliance collapses under homoglyph pressure
lr_shap_drift = np.mean(np.abs(shap_values_clean - shap_values_adv))
svc_shap_drift = np.mean(np.abs(shap_values_svc_clean.values - shap_values_svc_adv.values))

print("\n" + "="*65)
print("          POST-HOC EXPLAINABILITY METRIC STABILITY SURVEY")
print("="*65)
print(f"Logistic Regression SHAP Drift Index: {lr_shap_drift:.6f}")
print(f"Linear SVC (SVM) SHAP Drift Index:    {svc_shap_drift:.6f}")
print("="*65)
print("Note: High drift scores provide mathematical evidence that visual")
print("homoglyphs introduce structural blinding at the feature layer.")

In [ ]:
# ==========================================================
# CELL 13: EVALUATING DEFENSIVE NORMALIZATION AND ECO-EFFICIENCY
# ==========================================================
# Uncomment and activate your baseline normalization logic
X_test_defended = X_test_adversarial.apply(apply_defensive_normalization)
X_test_defended_tfidf = tfidf.transform(X_test_defended)

# Evaluate defensive restoration
preds_defended_lr = lr_model.predict(X_test_defended_tfidf)
f1_defended_lr = f1_score(y_test, preds_defended_lr)

preds_defended_svc = svc_model.predict(X_test_defended_tfidf)
f1_defended_svc = f1_score(y_test, preds_defended_svc)

print("\n" + "="*70)
print("              SECURITY RESTORATION VS SUSTAINABILITY ANALYSIS")
print("="*70)
print(f"Logistic Regression F1 Recovery: {f1_clean:.4f} (Clean) -> {f1_adv:.4f} (Attacked) -> {f1_defended_lr:.4f} (Protected)")
print(f"Linear SVC (SVM) F1 Recovery:    {f1_clean_svc:.4f} (Clean) -> {f1_adv_svc:.4f} (Attacked) -> {f1_defended_svc:.4f} (Protected)")
print("="*70)